# A small CNN on shape images

Images are tensors shaped `(batch, channels, height, width)`. A convolutional network slides small filters over the image to detect local patterns such as edges. The task here is to tell **horizontal** bars from **vertical** bars in noisy 16x16 images.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn

from pytorch_examples.data import make_loader, make_shape_images, train_val_split
from pytorch_examples.models import SmallCNN
from pytorch_examples.train import accuracy, fit
from pytorch_examples.utils import count_parameters, set_seed

set_seed(0)
images, labels = make_shape_images(n=256, size=16, noise=0.3, seed=0)
print("images:", tuple(images.shape), " labels:", tuple(labels.shape))

In [ ]:
names = {0: "horizontal", 1: "vertical"}
fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for ax, image, label in zip(axes.flat, images, labels):
    ax.imshow(image[0], cmap="gray")
    ax.set_title(names[int(label)], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Train

In [ ]:
x_tr, y_tr, x_val, y_val = train_val_split(images, labels, val_fraction=0.2, seed=0)

model = SmallCNN(in_channels=1, num_classes=2, image_size=16)
print("parameters:", count_parameters(model))

history = fit(
    model,
    make_loader(x_tr, y_tr, batch_size=32, seed=0),
    make_loader(x_val, y_val, batch_size=64, shuffle=False),
    loss_fn=nn.CrossEntropyLoss(),
    epochs=12,
    lr=5e-3,
    device="cpu",
    metric_fn=accuracy,
    verbose=True,
)

## Look at some predictions

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(x_val[:12]).argmax(dim=1)

fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for ax, image, truth, guess in zip(axes.flat, x_val[:12], y_val[:12], predictions):
    ax.imshow(image[0], cmap="gray")
    ax.set_title(f"{names[int(guess)]}", fontsize=8, color="green" if truth == guess else "red")
    ax.axis("off")
plt.suptitle("Predictions (green = correct)")
plt.tight_layout()
plt.show()

In [ ]:
final_accuracy = history.val_metric[-1]
print(f"validation accuracy: {final_accuracy:.2f}  (chance is 0.50)")
assert final_accuracy > 0.7